# 8. RAG (Retrieval Augmented Generation)

*Goal: Search our dataset using natural language.*

## Step 1 — Import Libraries

Import the required modules:
- `uuid` for unique id generation
- `pandas` for data loading
- `SecretStr` from `pydantic` to safely wrap the API key
- `InMemoryVectorStore` from `langchain_core.vectorstores` as the in-memory vector store
- `Document` from `langchain_core.documents` to wrap each command with its metadata
- `ChatPromptTemplate` from `langchain_core.prompts` for building the prompt template
- `StrOutputParser` from `langchain_core.output_parsers` to parse the LLM output to a string
- `RunnablePassthrough` from `langchain_core.runnables` to pass the question through the chain unchanged

## Step 2 — Load the Dataset

Load **`_06_unique_clustered_commands_with_risk.csv`** into a variable called `dataframe` — the enriched dataset from notebook 6, containing unique deduplicated commands with their cluster descriptions and risk scores.

## Step 3 — Initialize the Embeddings Client
Create an embeddings client that matches the method used to generate the embeddingss and assign it to the `embeddings_client` variable. The nice thing about using LangChain is that it makes it trivial to adjust AI providers. Depending on the provider you are using, import the appropriate Chat class.


- OpenAI `from langchain_openai import OpenAIEmbeddings`
- Google AI Studio `from langchain_google_genai import GoogleGenerativeAIEmbeddings`
- Ollama `from langchain_ollama import OllamaEmbeddings`

Both OpenAI and Google will require API keys, but if you are running Ollama locally, you will not need an API key. Here are examples of creating a chat client for these providers.

```python
OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=SecretStr(api_key)
)
```

```python
GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    api_key=SecretStr(api_key)
)
```

```python
OllamaEmbeddings(
    model="embeddinggemma"
)
```

Related Docs:
- https://docs.langchain.com/oss/python/integrations/embeddings/openai
- https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai
- https://docs.langchain.com/oss/python/integrations/embeddings/ollama


## Step 4 — Initialize the Chat Client
Initialize a Chat client using one of the following methods and assign it to a `chat_model` variable. The nice thing about using LangChain is that it makes it trivial to adjust AI providers. Depending on the provider you are using, import the appropriate Chat class.


- OpenAI `from langchain_openai import ChatOpenAI`
- Google AI Studio `from langchain_google_genai import ChatGoogleGenerativeAI`
- Ollama `from langchain_ollama import ChatOllama`

Both OpenAI and Google will require API keys, but if you are running Ollama locally, you will not need an API key. Here are examples of creating a chat client for these providers. 

```python
ChatOpenAI(
    model="gpt-4o-mini",
    api_key=SecretStr(api_key),
)
```

```python
ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    api_key=SecretStr(api_key),
)
```

```python
ChatOllama(
    model="gemma2",
)
```

Related Docs:
- https://docs.langchain.com/oss/python/integrations/chat/openai
- https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
- https://docs.langchain.com/oss/python/integrations/chat/ollama


## Step 5 — Build LangChain Documents

Build a list of LangChain `Document` objects by iterating over `dataframe.iterrows()` using a list comprehension (`for _, row in dataframe.iterrows()`). Store the result in a variable named `documents`.

For each row, create a `Document` with:
- `page_content` set to `row["CommandLine"]`
- `metadata` as a dictionary with three keys:
  - `"cluster"` — `int(row["Cluster_DBSCAN"])`
  - `"description"` — `row["DBSCANClusterDescription"]`
  - `"risk_score"` — `int(row["DBSCANClusterRiskScore"])`

## Step 6 — Create the Vector Store

In this step you will build an in-memory vector store using the **pre-computed embeddings** already stored in `dataframe["Embedding"]`.

### 6a — Initialize an Empty In-Memory Vector Store

Create the vector store using `InMemoryVectorStore(embedding=embeddings_client)` and assign it to `vector_store`.

Related Docs:
 - https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore

### 6b — Insert Documents with Their Existing Embeddings
This part is a little bit of a hack. Because we have already calculated the embeddings and don't
want to regenerate them, we will manually set each document using `InMemoryVectorStore` internal variables.
Iterate over `zip(documents, dataframe["Embedding"])` so each `Document` is paired with its corresponding vector.

Example:
```python
for doc, vector in zip(documents, dataframe["Embedding"]):
```

Inside the loop:
- set `doc_id = doc.id or str(uuid.uuid4())`. This will set a unique document id.
- add an entry to `vector_store.store[doc_id]` with keys (this is the hacky part):
  - `"id"`: the `doc_id`
  - `"vector"`: the embedding vector from the DataFrame
  - `"text"`: `doc.page_content`
  - `"metadata"`: `doc.metadata`

Example:
```python
vector_store.store[doc_id] = {
  "id": doc_id,
  "vector": vector,
  "text": doc.page_content,
  "metadata": doc.metadata,
}
```

## Step 7 — Build the RAG Chain

Assemble the RAG pipeline using LangChain Expression Language (LCEL):

1. **Retriever** — fetches the top-`k` most similar commands from vector store for a given question
2. **`format_docs`** — formats the retrieved documents into a readable context block, including each command's cluster, description, and risk score
3. **Prompt** — injects the formatted context and the analyst's question into a DFIR-focused system prompt
4. **LLM** — generates a grounded answer using only the retrieved context
5. **`StrOutputParser`** — extracts the final answer string from the LLM response

> LCEL chains use the `|` operator to compose steps, making the data flow explicit and easy to modify.

### 7a - Create the Retriever

Create a variable named `retriever` using `vector_store.as_retriever(...)`.
Set `search_kwargs` so `k` is `100` to retrieve the top 100 similar commands.

Related Docs:
 - https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/as_retriever
 

### 7b - Define the Formatter Function

Define a function named `format_docs` that accepts one parameter: `docs`.
Add a short docstring explaining that it formats retrieved documents into a readable context block.

Example:
```python
def format_docs(docs):
    """Format retrieved documents into a readable context block."""
    entries = []
    for doc in docs:
        entries.append(
            f"{doc.page_content}"
        )
    return "\n\n".join(entries)
```

### 7c - Build the Prompt Template

Create a variable named `prompt` using `ChatPromptTemplate.from_messages([...])`.
Include a system message that enforces DFIR analyst behavior and uses `{context}`.
Include a human message that uses `{question}`.

Example:
```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "Retrieved commands:\n{context}"),
    ("human", "{question}")
])
```

Related Docs:
 - https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate

### 7d - Create a RAG pipeline for querying

Build a chain named `rag_chain` that:
1. Uses the retriever output (formatted with `format_docs`) as `context`.
2. Passes the user question through as `question`.
3. Pipes those values into `prompt`, then `chat_model`, then `StrOutputParser()`.

Tip: Use LCEL pipe syntax (`|`) and `RunnablePassthrough()` for the raw question.

Code snippet:
```python
rag_chain = (
    {"context": retriever | format_docs, 
     "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)
```

Related Docs:
 - https://reference.langchain.com/python/langchain-core/runnables/passthrough/RunnablePassthrough
 - https://reference.langchain.com/python/langchain-core/output_parsers/string/StrOutputParser
 - https://reference.langchain.com/python/langchain-core/runnables/base/Runnable

## Step 8 — Query the RAG System

Ask natural-language questions about the commands in the dataset. The chain will:
1. Embed the question using embeddings model
2. Retrieve the 100 most semantically similar commands from the vector store
3. Feed the retrieved context and the question to LLM chat model
4. Return an answer based of results from the retrieval

Example:
```python
question = input("Ask a question about the commands in the dataset: ")
answer = rag_chain.invoke(question)
```

Try questions such as:
- *"Which commands are related to credential dumping?"*
- *"What lateral movement techniques were observed?"*
- *"Are there any high-risk commands related to defence evasion?"*